# Test a mesh-transformer checkpoint

Loads the **same** test split a training run used -- same config, same seed, same
`random_split` -- generates LOD2 from each LOD1 condition, and shows
input / ground truth / generated side by side.

Works for both tokenizers: a `coord` run predicts 9 coordinates per triangle, a
`vqvae` run (stage 2, `configs/mesh3-train.yaml`) predicts `depth` codes per
triangle through a frozen stage-1 codebook. The config decides; nothing below is
per-tokenizer except the generation budget.

Generation has no KV cache: one forward pass per token, so keep `N_SHOW` small.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import torch

from src.dataset.mesh_datamodule import MeshDataModule
from src.dataset.mesh_dataset import detokenize, mesh_collate_fn, specials
from src.eval.mesh_metrics import mesh_metrics
from src.models.mesh_transformer import MeshTransformerModule
from src.train_mesh import _load_vqvae
from src.utils.initialization import load_config
from src.visualize_mesh import cityjson_figure_from_mesh, mesh_figure, side_by_side

import plotly.io as pio

pio.renderers.default = "notebook" 

## The experiment to test

`CKPT = None` picks the newest checkpoint of the run the config names.

`VQVAE_CKPT` only matters for a `tokenizer: vqvae` config, where `mesh3-train.yaml`
deliberately ships `vqvae_ckpt: null` -- point it at the stage-1 checkpoint that
run trained against. A different codebook decodes the same codes into a different
mesh, so this must be the *same* file the run used.

In [ ]:
CONFIG = ROOT / "configs" / "mesh3-train.yaml"
CKPT = None          # None = newest .ckpt under the run directory
VQVAE_CKPT = ROOT / "outputs"/"initial-runs"/"mesh-vqvae-3"/"checkpoints"/"last.ckpt"
N_SHOW = 4           # buildings to generate; each one is slow
TEMPERATURE = 0.0    # 0 = argmax, as in mesh_eval

cfg = load_config(CONFIG, [])
run_dir = ROOT / cfg.logging.save_dir / cfg.logging.experiment_name / cfg.logging.run_name

if CKPT is None:
    ckpts = sorted(run_dir.rglob("*.ckpt"), key=lambda p: p.stat().st_mtime)
    if not ckpts:
        raise FileNotFoundError(f"No .ckpt under {run_dir} -- set CKPT explicitly.")
    CKPT = ckpts[-1]

# Paths in the config are relative to the repo root, not to notebooks/.
if cfg.mesh_model.tokenizer == "vqvae":
    ckpt = VQVAE_CKPT or cfg.mesh_model.vqvae_ckpt
    if not ckpt:
        raise ValueError("tokenizer='vqvae' needs a stage-1 checkpoint: set VQVAE_CKPT.")
    cfg.mesh_model.vqvae_ckpt = str(ROOT / ckpt)

print(f"config {CONFIG.name}, run {cfg.logging.run_name}, "
      f"tokenizer {cfg.mesh_model.tokenizer}")
print(f"checkpoint {Path(CKPT).relative_to(ROOT)}")

## The same test split

Every argument below is what `src/train_mesh.py` passes, `cfg.seed` included, so
`random_split` reproduces the exact partition the run trained against.

In [3]:
datamodule = MeshDataModule(
    dataset_dir=ROOT / cfg.mesh_data.dataset_dir,
    lod_in=cfg.mesh_data.lod_in,
    lod_out=cfg.mesh_data.lod_out,
    num_bins=cfg.mesh_data.num_bins,
    margin_lo=list(cfg.mesh_data.margin_lo),
    margin_hi=list(cfg.mesh_data.margin_hi),
    max_faces=cfg.mesh_data.max_faces,
    max_files=cfg.mesh_data.max_files,
    batch_size=cfg.training.batch_size,
    train_val_test_split=tuple(cfg.training.train_val_test_split),
    num_workers=0,
    seed=cfg.seed,
)
datamodule.setup()
test_set = datamodule.test_dataset

print(f"train={len(datamodule.train_dataset)} val={len(datamodule.val_dataset)} "
      f"test={len(test_set)}")

train=16032 val=2004 test=2005


In [ ]:
# The tokenizer is not a hyperparameter (save_hyperparameters ignores it), so it
# has to be rebuilt and handed in -- otherwise the checkpoint's `vqvae.*` weights
# have nowhere to load. `_load_vqvae` is what train_mesh.py uses, so the
# architecture and the num_bins check are identical.
vqvae = _load_vqvae(cfg)
model = MeshTransformerModule.load_from_checkpoint(CKPT, map_location="cpu", vqvae=vqvae)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Same budget rule as MeshEvalCallback: asking past the positional embedding raises.
# Under the VQ-VAE that is 3 vertices x depth codes per face, which at the
# paper's depth of 3 is the same 9 tokens the coordinate tokenizer uses.
per_face = 9 if model.vqvae is None else model.vqvae.tokens_per_face
max_new_tokens = min(per_face * (cfg.mesh_data.max_faces or 200) + 1,
                     model.network.max_seq_len - 1)
print(f"{model.num_bins} bins, {per_face} tokens/face, "
      f"max_seq_len {model.network.max_seq_len}, budget {max_new_tokens} tokens on {device}")

## Generate

In [ ]:
def to_metres(mesh, center, scale):
    """(verts, faces) in the unit box -> metres, undoing the LOD1-box frame."""
    verts, faces = mesh
    return verts * scale + center, faces


items = [test_set[i] for i in range(min(N_SHOW, len(test_set)))]
# The dataset speaks coordinates whatever the model speaks, so pad with the
# coordinate PAD -- `_prepare` re-encodes and re-pads on the VQ-VAE path.
batch = mesh_collate_fn(items, pad=specials(model.num_bins)[2])
batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

with torch.no_grad():
    # Not batch["cond"] directly: under the VQ-VAE the condition is a continuous
    # face feature, not token ids, and `tgt` is codes rather than coordinates.
    cond, tgt, cond_pad, _ = model._prepare(batch)
    out = model.generate(cond, cond_pad,
                         max_new_tokens=max_new_tokens, temperature=TEMPERATURE)

results = []
for k, name in enumerate(batch["ids"]):
    center = batch["center"][k].cpu().numpy()
    scale = batch["scale"][k].cpu().numpy()
    # The condition is always coordinate tokens; ground truth and generation go
    # through whichever tokenizer the model was built with, so what is left
    # between them is model error and not tokenizer loss.
    lod1 = to_metres(detokenize(batch["cond"][k].cpu().numpy(), model.num_bins), center, scale)
    gt = to_metres(model.decode_tokens(tgt[k]), center, scale)
    gen = to_metres(model.decode_tokens(out[k]), center, scale)
    results.append((name, lod1, gt, gen))
    print(f"{name}: lod1 {len(lod1[1])} tris | gt {len(gt[1])} | gen {len(gen[1])}")

## Input / ground truth / generated

Top row: the raw triangle meshes. The LOD1 input is the coordinate round trip (it
is the condition, never decoded by the model); ground truth and generation both
come back through the model's own tokenizer, so what is left between the middle
and right panel is model error alone. Under `tokenizer: vqvae` the middle panel is
therefore the *codebook's* best reconstruction of the true LOD2, not the true LOD2
-- the gap between it and the raw mesh is the stage-1 ceiling.

Bottom row: the same geometry through `mesh_to_cityjson` -- coplanar triangles merged
back into polygons and coloured by semantic surface (blue ground, orange roof, grey
wall). An empty bottom panel means the writer refused the mesh.

In [7]:
from IPython.display import display

figs = []

for name, lod1, gt, gen in results:
    row = mesh_metrics(gen, gt, taus=tuple(cfg.mesh_eval.taus),
                       n_points=cfg.mesh_eval.n_points, voxel_m=cfg.mesh_eval.voxel_m)
    print(name, {k: round(v, 4) for k, v in sorted(row.items()) if np.isfinite(v)})

    meshes = [(lod1, "#898781"), (gt, "#2a78d6"), (gen, "#eda100")]
    top = [mesh_figure(*mesh, color=color) for mesh, color in meshes]
    bottom = [cityjson_figure_from_mesh(*mesh) for mesh, _ in meshes]

    # display(), not fig.show(): show() picks a renderer and draws nothing when
    # it guesses wrong, while display() emits the same mime bundle a bare figure
    # at the end of a cell does -- which is what already renders elsewhere.
    figs.append(side_by_side(
        [top, [fig for fig, _ in bottom]],
        [f"LOD1 input, {len(lod1[1])} tris",
         f"LOD2 ground truth, {len(gt[1])} tris",
         f"LOD2 generated, {len(gen[1])} tris"]
        + [f"CityJSON, {n} surfaces" if n else "CityJSON: none" for _, n in bottom],
    ))

NL.IMBAG.Pand.0626100000006685-0 {'chamfer_m': 0.0553, 'footprint_iou': np.float64(1.0), 'fscore_25cm': 1.0, 'fscore_50cm': 1.0, 'hausdorff_p95_m': 0.1059, 'roof_max_z_err_m': 0.0387, 'roof_mean_z_err_m': 0.0132, 'watertight_gen': 0.0, 'watertight_gt': 1.0}
bag_0518100000286136 {'chamfer_m': 0.2205, 'footprint_iou': np.float64(0.9988), 'fscore_25cm': 0.7695, 'fscore_50cm': 0.9476, 'hausdorff_p95_m': 0.8973, 'roof_max_z_err_m': 0.0, 'roof_mean_z_err_m': 0.14, 'watertight_gen': 0.0, 'watertight_gt': 1.0}
NL.IMBAG.Pand.1901100000022372-0 {'chamfer_m': 0.3454, 'footprint_iou': np.float64(0.9988), 'fscore_25cm': 0.7082, 'fscore_50cm': 0.8027, 'hausdorff_p95_m': 1.5568, 'roof_max_z_err_m': 0.0, 'roof_mean_z_err_m': 1.4135, 'watertight_gen': 0.0, 'watertight_gt': 1.0}
NL.IMBAG.Pand.0603100000012423-0 {'chamfer_m': 0.3055, 'footprint_iou': np.float64(0.9985), 'fscore_25cm': 0.7108, 'fscore_50cm': 0.8199, 'hausdorff_p95_m': 1.2918, 'roof_max_z_err_m': 0.1733, 'roof_mean_z_err_m': 0.1703, 'water

In [11]:
figs[3].show()